In [1]:
sc

<SparkContext master=local[*] appName=PySparkShell>

In [2]:
spark

ImportError: PyArrow >= 1.0.0 must be installed; however, it was not found.

In [3]:
# pip install pyarrow

Successfully installed pyarrow-7.0.0

In [4]:
# pyarrow.__version__ # NameError: name 'pyarrow' is not defined
# python.__version__ # NameError: name 'python' is not defined
# hadoop.__version__ # NameError: name 'hadoop' is not defined


In [5]:
import numpy as np
from PIL import Image
import io
from tensorflow.keras.preprocessing.image import img_to_array
import pandas as pd


2022-03-16 17:11:03.755848: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2022-03-16 17:11:03.755889: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [6]:
from pyspark.sql import SparkSession

In [7]:
from tensorflow.keras.applications.mobilenet import preprocess_input, MobileNet
# from tensorflow.keras.applications.resnet50 import preprocess_input, ResNet50 


In [8]:
from pyspark.sql.functions import size

In [9]:
from pyspark.sql.functions import col, pandas_udf, PandasUDFType

In [10]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

In [11]:
from pyspark.ml.feature import PCA

In [12]:
# from pyspark.ml.linalg import Vectors, VectorUDT

In [13]:
from pyspark.ml.functions import array_to_vector

In [14]:
from pyspark.ml.feature import StandardScaler

In [15]:
from pyspark.ml.functions import vector_to_array

# Load Images into a Spark DataFrame

To get access to S3 bucket, you have to establish Spark Session first, and send to the bucket the key pair.
The codeline precising the endpoint is odd, it breaks the connection, so is commented out. When this line is executed, it causes the eternal suspension of any cell related to SparkSession. The endpoints set by default are correct, you don't need to precise them.

In [16]:
spark = SparkSession.builder.appName("app_name1").getOrCreate()
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.access.key", "TheKey1SensitiveInformation")
#hadoop_conf.set("fs.s3a.awsAccessKeyId", "TheKey1SensitiveInformation")
hadoop_conf.set("fs.s3a.secret.key", "TheKey2SensitiveInformation")
#hadoop_conf.set("fs.s3a.awsSecretAccessKey", "TheKey2SensitiveInformation")

hadoop_conf.set("fs.s3a.impl","org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("com.amazonaws.services.s3.enableV4", "true")
hadoop_conf.set("fs.s3a.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
#hadoop_conf.set("fs.s3a.endpoint", "eu-central-1.amazonaws.com")  # CAUSES ERROR/SUSPENSION

In [17]:
spark

## The access to the files:
for the real work, indicate the folder name and set the recursiveFileLookup to True. This configuration open every folder and subfolder recursively to as many embedded folders as you have and extracts files from all the subfolder.  With my test set, that I have downloaded into S3 bucket (two folders and 6 free images), it gives about 400 files on every load.  
  
For the test use in free option, I indicate the folder name and * .jpg which means only jpg files, no folder. This is done in order to save money : in Free Tier you have only 20,000 GET requests and ony 2,000 PUT and LIST requests.
For the pricing policy, see https://aws.amazon.com/fr/free/ and then price-list for Frankfurt server (the closest to Strasbourg).

In [18]:
#   .load('/home/vb/Pictures/*.jpg')# (6, 4) only images out of the folder
#   .load('/home/vb/Pictures/')# (338, 4) images in Pictures and all the inner folders - if you use the .option("recursiveFileLookup", "true") \

Load images using Spark's binary file data source. You could alternatively use Spark's image data source, but the binary file data source provides more flexibility in how you preprocess images.

In [19]:
images = spark.read.format("binaryFile") \
  .option("pathGlobFilter", "*.jpg") \
  .option("recursiveFileLookup", "true") \
  .load("s3a://p8-s3-bucket/")

#   .load("s3a://p8-s3-bucket/*.jpg")# (6, 4)
#   .load("s3a://p8-s3-bucket/") # all the jpg images in all the folders and subfolders. I downloaded 338


print(('count', images.count(), 'columns', len(images.columns)))  # (6, 4)
images.printSchema() # schema only
display(images.limit(5))

22/03/16 17:11:11 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


('count', 338, 'columns', 4)
root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)



DataFrame[path: string, modificationTime: timestamp, length: bigint, content: binary]

# Repartitions

Repartitions allow parallel calculations on several VCP (processor units). As the Free Tier configuration t2.micro provides only 1 VCP, for study project it makes no sense to make a repartition. It will only consume more RAM, that I'm already heavily limited.  
If I did repartitions, the results are esay to collect with the function .collect()

In [20]:
# # repartition dataframe
# images = images.repartition(4) # if I have 4 VCP and many images
# images.printSchema()

In [21]:
# images # DataFrame[path: string, modificationTime: timestamp, length: bigint, content: binary]

# Import the model

The list of availables models is here:  
https://keras.io/api/applications/  

In real work, we would probably prefer the models with highest top-5 accuracy (EfficientNetV2L) or highest top-1 accuracy(EfficientNetV2L too). But this neural network alone eats 479 Mb of RAM, while I have already 475 Mb used after Spark Session opening and 609 Mb used aflet loading a PySpark Schema.  
I could use a compromise variant, like ResNet50 or InceptionV3 neural networks, which weight 98 and 92 Mb. But even these neural networks make my tiny free server  crash.  
I tried with the MobileNetV2 and MobileNet, where first has the smallest size (14 Mb) and second has slighlty bigger size (16 Mb) but the shortest time per inference step on CPU (22.6 ms).  

In [22]:
preprocess_input

<function keras.applications.mobilenet.preprocess_input(x, data_format=None)>

In [23]:
model = MobileNet(include_top=False, input_shape=(32, 32, 3), pooling='avg')
# model = ResNet50 (include_top=False, input_shape=(100, 100, 3), pooling='max')
# model = ResNet50(include_top=False)
model.summary()  # verify that the top layer is removed

# EXTRACT FROM MODEL SUMMARY: last layer is conv5_block3_out (Activation)  (None, None, None, 2048) 
# the output of the last layer is a vector, or a [vector], or [[vector]], or [[[vector]]] of length 2048
# model

# # # conv_pw_13_relu (ReLU)      (None, 3, 3, 1024)        0         
                                                                 
# # #  global_average_pooling2d_4   (None, 1024)             0         
# # #  (GlobalAveragePooling2D)

2022-03-16 17:11:26.885789: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2022-03-16 17:11:26.886151: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2022-03-16 17:11:26.886180: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (ip-172-31-29-101): /proc/driver/nvidia/version does not exist
2022-03-16 17:11:26.902683: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Model: "mobilenet_1.00_224"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 32, 32, 3)]       0         
                                                                 
 conv1 (Conv2D)              (None, 16, 16, 32)        864       
                                                                 
 conv1_bn (BatchNormalizatio  (None, 16, 16, 32)       128       
 n)                                                              
                                                                 
 conv1_relu (ReLU)           (None, 16, 16, 32)        0         
                                                                 
 conv_dw_1 (DepthwiseConv2D)  (None, 16, 16, 32)       288       
                                                                 
 conv_dw_1_bn (BatchNormaliz  (None, 16, 16, 32)       128       
 ation)                                         

In [24]:
from tensorflow.keras import layers, models


# source_model = MobileNet(include_top=True, input_shape=(224, 224, 3), pooling='avg')
source_model = MobileNet(include_top=True, pooling='avg')
my_model = models.Sequential()

for layer in source_model.layers[:-1]:
    my_model.add(layer)    

# Freeze the layers 
for layer in my_model.layers:
    layer.trainable = False

my_model.summary() # start not with InputLayer, as in source model, but with conv1 layer

# I have to rewrite all the functions in hte "Functions" block to ensure the correst interaction,
# or add the input layer to my_model and set its shape.

# I keep this approach as a plan B, if image resizing doesn't work.

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1 (Conv2D)              (None, 112, 112, 32)      864       
                                                                 
 conv1_bn (BatchNormalizatio  (None, 112, 112, 32)     128       
 n)                                                              
                                                                 
 conv1_relu (ReLU)           (None, 112, 112, 32)      0         
                                                                 
 conv_dw_1 (DepthwiseConv2D)  (None, 112, 112, 32)     288       
                                                                 
 conv_dw_1_bn (BatchNormaliz  (None, 112, 112, 32)     128       
 ation)                                                          
                                                                 
 conv_dw_1_relu (ReLU)       (None, 112, 112, 32)      0

In [25]:
source_model.summary() 

Model: "mobilenet_1.00_224"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 conv1 (Conv2D)              (None, 112, 112, 32)      864       
                                                                 
 conv1_bn (BatchNormalizatio  (None, 112, 112, 32)     128       
 n)                                                              
                                                                 
 conv1_relu (ReLU)           (None, 112, 112, 32)      0         
                                                                 
 conv_dw_1 (DepthwiseConv2D)  (None, 112, 112, 32)     288       
                                                                 
 conv_dw_1_bn (BatchNormaliz  (None, 112, 112, 32)     128       
 ation)                                         

What happens in reality, I get not the result of the last average pooling layer, but the beforelast conv_pw_13 layer.  
Thus I have the output size not 1024, but 9 times more, 9216. And this length of the vector is too large for my tiny server with little RAM.  
So, I will take the full model and remove the last layer (predictions).

In [26]:
bc_model_weights = sc.broadcast(model.get_weights())
# bc_model_weights = sc.broadcast(my_model.get_weights())
bc_model_weights

# Functions to ensure the parts' interaction

In [27]:
def model_fn():
    """
    Returns a MobileNet model with top layer removed and broadcasted pretrained weights.
    """
    model = MobileNet(weights=None, include_top=False)
#     model = ResNet50(weights=None, include_top=False)
    model.set_weights(bc_model_weights.value)
    return model

In [28]:
def preprocess(content):
    """
    Preprocesses raw image bytes for prediction.
    """
    
    img = Image.open(io.BytesIO(content)).resize([32, 32])
    arr = img_to_array(img)
    return preprocess_input(arr)

In [29]:
def featurize_series(model, content_series):
    """
    Featurize a pd.Series of raw images using the input model.
    :return: a pd.Series of image features
    """
    my_input = np.stack(content_series.map(preprocess))
    print('type my_input', type(my_input))
    print('my input shape', my_input.shape)    
    preds = model.predict(my_input) # shows not the last pooling layer, but the beforelast 3*3*1024 conv layer
    print('type preds', type(preds))
    print('preds shape', preds.shape)
# For some layers, output features will be multi-dimensional tensors.
# We flatten the feature tensors to vectors for easier storage in Spark DataFrames.
    output = [p.flatten() for p in preds]
    print('type output', type(output))
    print('len output', len(output))
    return pd.Series(output)

model.predict(my_input)  shows not the last pooling layer, but the beforelast 3*3*1024 conv layer.  

How to overcome this obstacle:  
- __reduce image size to, say, 32 * 32 pixels__ (that's my choice by now)
- use another neuron network instead of MobileNet, that outputs flat vector at the same code implementation. Say, ResNet50 (takes more RAM).
- compose your own model with Mobile Net. How to do so :
    - create a source MobileNet instance with include top = True (CAUTION: in this case the input shape can be only 224 * 224)
    - create a new Sequential instance (empty model)
    - copy layers one by one from source model to custom model (CAUTION: in this case the custom model starts from conv1 layer, while source model starts with input layer
- add manually a layer to the MobileNet model. As we see here, pooling layer doesn't process the vector here, even though it is present in model summary. So, we can add a layer that goes in a model after pooling. It is dropout (Dropout). These are still features. Next  layer, conv_preds (Conv2D) converts features to the classes to predict, and it's no good for our purposes in this project. And last 2 layers, reshape_2 (Reshape) and predictions (Activation), work already with the predictions, not with features.

In [30]:
@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    '''
    This method is a Scalar Iterator pandas UDF wrapping our featurization function.
    The decorator specifies that this returns a Spark DataFrame column of type ArrayType(FloatType).
  
    :param content_series_iter: This argument is an iterator over batches of data, 
    where each batch is a pandas Series of image data.
    '''
# With Scalar Iterator pandas UDFs, we can load the model once and then re-use it for multiple data batches.  
# This amortizes the overhead of loading big models.


    model = model_fn() # see whether it changes the output shape. UPD: no, only slows down the calculation
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)

/home/ubuntu/spark-3.2.1-bin-hadoop3.2/python/pyspark/sql/pandas/functions.py:389: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(


# Preprocessing step: apply featurization to the DataFrame of images

In [31]:
# Pandas UDFs on large records (e.g., very large images) can run into Out Of Memory (OOM) errors.
# If you hit such errors in the cell below, try reducing the Arrow batch size via `maxRecordsPerBatch`.

# it makes sense in the real project with many images. Here I have 6 images in total.
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "1024")

In [32]:
# We can now run featurization on our entire Spark DataFrame.
# NOTE: This can take a long time (about 10 minutes) since it applies a large model to the full dataset.
# NOTE: I don't have to use repartitions in this implementation, because I have only 1 vCPU on the free configuration of the server

# features_df = images.repartition(16).select(col("path"), featurize_udf("content").alias("features"))
features_df = images.select(col("path"), featurize_udf("content").alias("features"))
features_df

DataFrame[path: string, features: array<float>]

INFO:tensorflow:Assets written to: ram://b25811e7-f41a-490f-815b-f8dfe89b401c/assets

In [33]:
print(('count', features_df.count(), 'columns', len(features_df.columns)))
features_df.printSchema()

('count', 338, 'columns', 2)
root
 |-- path: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = true)



I wonder why the "element" sub-column contains NULL, if it is the output of the neural network, so it's not expected to have any NULL values.

In [34]:
# features_df.filter("features is NULL").show() # empty PySpark DataFrame

No rows with features are null, but I want to know if any array contains NULL values

To know the length of the resulting array:

In [35]:
# features_df.select(size(features_df.features)).collect()


# # OUTPUT  [Row(size(features)=9216), Row(size(features)=9216)] 

# # [Row(size(features)=1024),
# #  Row(size(features)=1024),
# #  Row(size(features)=1024),
# #  Row(size(features)=1024),
# #  Row(size(features)=1024),
# #  Row(size(features)=1024)]

# # 338 * 1024

The length of the output vector in the array is 9216.  
Worth mentioning that for ResNet50, a good compromise solution, the length of the output vector is 2048. And for VGG 16 it is 4096.

After using swap of 4 Gb, the ResNet50 network could be applied.  
And the output is not 2048 as expected, but 32768. So I returned to MobileNet.

In [36]:
# features_df.select("path", "features.element").show(truncate=True) # nalysisException: cannot resolve 'features['element']' due to data type mismatch: argument 2 requires integral type, however, ''element'' is of string type.;

# features_df.select("path", "features").show(truncate=True)  # works all the same
# features_df.show(truncate=True) # works all the same
# only at this step the execution starts, and the server craches UPD: after applying swap it's fine

# Dimension reduction step with PCA

## WARNING : number of components instead of variance
When doing PCA in Scikit Learn, you can either set the desired number of components K, if you insert an integer number, or you can indicate the desired degree of preserved variance, if you indicate a float number, say, 0.99.
In PySpark you don't have the option to select desired level of preserved variance. You can only indicate the number of components.
https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.PCA.html#pyspark.ml.feature.PCA

## WARNING: n_components == min(n_samples, n_features) - 1

It makes sence: PCA is a projection of N data points in P-dimensional space into Q-dimensional space where Q < P.
But if the space dimension P is higher that the number of points N, the projection has no solution.  
Consider the example of 2 dots in 3-D space. You want to make a projection into 2-D space, but you have infinite number of planes preserving the maximal variance: these are the planes containing the line linking the two dots.  
But if you want to project them into 1-D space, it's fine: the solution is unique and it is the line connecting the 2 dots.  
As I have downloaded 6 images into the bucket, I suppose I can use up to 5 components, but no more.  
Nevertheless, I will try to use a PCA with 20 components and see what it gives: whether the code crashes or not.

Column features must be of type class org.apache.spark.ml.linalg.VectorUDT:struct < type:tinyint, size:int, indices:array < int > , values:array < double > > but was actually class org.apache.spark.sql.types.ArrayType:array < float >.

In [37]:
features_df = features_df.withColumn('vector_features', array_to_vector('features'))
features_df.printSchema()

root
 |-- path: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- vector_features: vector (nullable = true)



In [38]:
# features_df.show() # ok

In [39]:
# scaler = StandardScaler(inputCol="vector_features", outputCol="vector_features", withStd=True, withMean=True)
# IllegalArgumentException: requirement failed: Output column vector_features already exists.

scaler = StandardScaler(inputCol="vector_features", outputCol="scaled_features", withStd=True, withMean=True)
scalerModel = scaler.fit(features_df)

2022-03-16 17:11:42.876041: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2022-03-16 17:11:42.883917: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2022-03-16 17:11:48.673762: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2022-03-16 17:11:48.674769: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2022-03-16 17:11:48.675621: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (ip-172-31-29-101): /proc/driver/nvidia/version does not exist
2022-03-16 17:11:48.681025: I tensorflow/core/platform/cpu_f

In [40]:
features_df = scalerModel.transform(features_df).drop('vector_features')
features_df.printSchema()

root
 |-- path: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- scaled_features: vector (nullable = true)



In [41]:
# features_df.show() # ok

In [42]:
komponents = 10
pca = PCA(k=komponents, inputCol="scaled_features", outputCol="principal_components")
pcaModel = pca.fit(features_df)

type my_input <class 'numpy.ndarray'>                               (0 + 1) / 1]
my input shape (32, 32, 32, 3)
type preds <class 'numpy.ndarray'>
preds shape (32, 1, 1, 1024)
type output <class 'list'>
len output 32
2022-03-16 17:12:24.321011: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2022-03-16 17:12:24.324282: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2022-03-16 17:12:29.636231: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2022-03-16 17:12:29.637038: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2022-03-16 17:12:29.637072: I tensorflow/stream_ex

In [43]:
sc._conf.get('spark.driver.memory')


'10g'

In [44]:
print("====Eigenvector====")
print(pcaModel.pc)

print("====Contribution rate====")
print(pcaModel.explainedVariance)

====Eigenvector====
DenseMatrix([[0., 0., 0., ..., 0., 0., 0.],
             [0., 0., 0., ..., 0., 0., 0.],
             [0., 0., 0., ..., 0., 0., 0.],
             ...,
             [0., 0., 0., ..., 0., 0., 0.],
             [0., 0., 0., ..., 0., 0., 0.],
             [0., 0., 0., ..., 0., 0., 0.]])
====Contribution rate====
[0.09976353793728654,0.08940339671013957,0.0694030651180799,0.06252175099732617,0.05604492192230124,0.052484636058596205,0.046139167588862676,0.0419139915678988,0.0370339585770565,0.035688709143704415]


In [45]:
features_df = pcaModel.transform(features_df).drop('scaled_features')
features_df.printSchema()

root
 |-- path: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- principal_components: vector (nullable = true)



In [46]:
# features_df.show() # ok

## Saving the principal components in a string format into a .csv file

In [47]:
features_df_TEST = features_df.withColumn('principal_components', vector_to_array('principal_components'))
features_df_TEST.printSchema()

root
 |-- path: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- principal_components: array (nullable = false)
 |    |-- element: double (containsNull = false)



In [48]:
# features_df_TEST.select("path", features_df_TEST.principal_components[0], features_df_TEST.principal_components[1], features_df_TEST.principal_components[2]).show()
# ok


But here we have presented only 3 columns out of 10.  
I can list all the 10 columns manually, but what if there is 100? 1,000? What if the columns number has ben modified again and again? Ican use list comprehension

In [49]:
['path', *[features_df_TEST.principal_components[i] for i in range(komponents)]]

['path',
 Column<'principal_components[0]'>,
 Column<'principal_components[1]'>,
 Column<'principal_components[2]'>,
 Column<'principal_components[3]'>,
 Column<'principal_components[4]'>,
 Column<'principal_components[5]'>,
 Column<'principal_components[6]'>,
 Column<'principal_components[7]'>,
 Column<'principal_components[8]'>,
 Column<'principal_components[9]'>]

In [50]:
# features_df_TEST.select("path", *[features_df_TEST.principal_components[i] for i in range(komponents)]).show()
# looks ugly but the results structure ok, makes sense, it's what I expect

In [51]:
features_df_TEST.select("path", *[features_df_TEST.principal_components[i] for i in range(komponents)]).write.mode("overwrite").csv("s3a://p8-s3-bucket/Principle_components_csv")
# 

22/03/16 17:14:12 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
22/03/16 17:14:13 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
type my_input <class 'numpy.ndarray'>                              (0 + 1) / 11]
my input shape (32, 32, 32, 3)
type preds <class 'numpy.ndarray'>
preds shape (32, 1, 1, 1024)
type output <class 'list'>
len output 32
22/03/16 17:14:20 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
type my_input <class 'numpy.ndarray'>                              (1 + 1) / 11]
my input shape (32, 32, 32, 3)
type preds <class 'numpy.ndarray'>
preds shape (32, 1, 1, 1024)
type output <class 'list'>
len output 32
22/03/16 17:14:25 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
type m

# Old code: tests, out-of-date approaches

In [52]:
STOP FURTHER EXECUTION

SyntaxError: invalid syntax (4288356501.py, line 1)

## Test saving to .parquet file

This is a standard output format for PySpark DataFrames. It is used only to test whether the writing into the S3 bucket works. This is a test saving mode, just to see whether the saving works at all. When this method works with the bucket, I can comment out this section and proceed with saving the csv file into the bucket.

In [77]:
# # features_df.write.mode("overwrite").parquet("/home/vb/Desktop/test2/fruit_features.parquet")
# # features_df.write.parquet("/home/vb/Desktop/test")
# features_df.write.mode("overwrite").parquet("s3a://p8-s3-bucket/Saved_features_parquet")

RuntimeError: Python in worker has different version 3.8 than that in driver 3.9, PySpark cannot run with different minor versions. Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

printenv PYSPARK_PYTHON    /usr/bin/python3  
printenv PYSPARK_DRIVER_PYTHON    jupyter  

what do I do in console:  
export PYSPARK_PYTHON=/usr/bin/python3  
export PYSPARK_DRIVER_PYTHON=/usr/bin/python3  

## Saving the principal components in a string format into a .csv file

The problem with the PySpark DataFrame is that the "features" column is an array of floats. And trying to write down an array riese an error.
The solution that I found is to convert the array to a string, and write the string to the resulting .csv file.

The problem with the PySpark DataFrame is that the "principal_components" column is a Dense Vector. And trying to write down an array riese an error.  
The solution that I found is to convert the Dense vector to an array, than the array to a string, and write the string to the resulting .csv file.

It depends whether it's required to write the principal components only (the most probable), the features only, or both.

In [88]:
# def array_to_string(my_list):
#     return '[' + ','.join([str(elem) for elem in my_list]) + ']'

In [89]:
# array_to_string_udf = udf(array_to_string, StringType())

In [90]:
# features_df.printSchema()

root
 |-- path: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- principal_components: vector (nullable = true)



In [91]:
# features_df_TEST = features_df.withColumn('principal_components', vector_to_array('principal_components'))
# features_df_TEST.printSchema()

root
 |-- path: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- principal_components: array (nullable = false)
 |    |-- element: double (containsNull = false)



In [92]:
# stringified_df = features_df_TEST.withColumn('features', array_to_string_udf(features_df_TEST["features"]))
# stringified_df = stringified_df.withColumn('principal_components', array_to_string_udf(features_df_TEST["principal_components"]))
# stringified_df.printSchema()
# # stringified_df.show(truncate=True) 

root
 |-- path: string (nullable = true)
 |-- features: string (nullable = true)
 |-- principal_components: string (nullable = true)



In [93]:
# features_df.show() # ok

## Writing the image path and principle components

In [94]:
# stringified_df.select(['path', 'principal_components']).write.mode("overwrite").csv("s3a://p8-s3-bucket/Principle_components_as_string_csv")

22/03/16 16:47:26 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
22/03/16 16:47:27 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
2022-03-16 16:47:29.177874: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2022-03-16 16:47:29.177973: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2022-03-16 16:47:37.777766: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2022-03-16 16:47:37.779902: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN

## Writing the image path and features (full size), if needed

In [55]:
# stringified_df.select(['path', 'features']).write.mode("overwrite").csv("s3a://p8-s3-bucket/Saved_features_csv") # ok

In [56]:
# ImageSchema.readImages('s3://p8-s3-bucket/Fig').withColumn("label", lit(0)) # AttributeError: '_ImageSchema' object has no attribute 'readImages'

# import spark # DON'T UNCOMMENT IT: the code works but it is something weird, and has no method "read"

# this code works but it is useless
# spark_test1 = (SparkSession.builder.appName("app_name1").getOrCreate())
# spark_test1

In [57]:
# # figure out whu Pyspark DataFrame creation doesn't work
# # UPD: apparently all the problem is in the line about the endpoint. Comment out or delete this line

# # df_test1 = spark_test2.read.format("image").load("s3://p8-s3-bucket/0_100.jpg") # Py4JJavaError: An error occurred while calling o41.load
# df_test1 = spark_test2.read.format("image").load("s3a://p8-s3-bucket/0_100.jpg") # Py4JJavaError: An error occurred while calling o41.load 

# print((df_test1.count(), len(df_test1.columns)))
# print(df_test1.printSchema())
# df_test1.select('image.nChannels', "image.width", "image.height", "image.data").show(truncate=True)

# # csvDf = spark.read.csv("s3a://path/to/files/*.csv")
# # jsonDf = spark.read.json("s3://path/to/files/*.json")
# #spark_test2.stop() # DON'T UNCOMMENT IT

In [58]:
# df_test_2=spark.read.format("image").load("s3a://p8-s3-bucket/*.jpg")
# print((df_test_2.count(), len(df_test_2.columns)))  # (6, 1)
# # print(df_test_2.printSchema()) # schema, then None (output of "print"?)
# df_test_2.printSchema() # schema only
# df_test_2.select('image.nChannels', "image.width", "image.height", "image.data").show(truncate=True) # smth like a table


In [59]:
# # also works, just a bit slower, because contains more images
# # it works, but I must comment out this section, because every PUT and LIST request is payed and I'm already above the free limit.

# df_test_3=spark_test2.read.format("image").load("s3a://p8-s3-bucket/Mango/*.jpg")
# print((df_test_3.count(), len(df_test_3.columns)))
# print(df_test_3.printSchema())
# df_test_3.select('image.nChannels', "image.width", "image.height", "image.data").show(truncate=True)

## Old way to featurize images with DeepImageFeaturizer

In [60]:
# from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# from pyspark.ml.classification import LogisticRegression
# from pyspark.ml import Pipeline
# from sparkdl import DeepImageFeaturizer

In [61]:
# featurizer1 = DeepImageFeaturizer(inputCol="image",
#                                  outputCol="features",
#                                  modelName="VGG16") # TypeError: Invalid param value given for param "modelName". not enough arguments for format string

In [62]:
# featurizer1 = DeepImageFeaturizer(inputCol="image",
#                                  outputCol="features",
#                                  modelName="InceptionV3")

## Get access to data without opennig a Spark Session

These tests were performed during the checks that the connection between EC2 instance and S3 bucket works.  
Now this code is all commented out for the cost reason, in order not to waste money. in Free Tier you have only 20,000 GET requests and ony 2,000 PUT and LIST requests.
For the pricing policy, see https://aws.amazon.com/fr/free/ and then price-list for Frankfurt server (the closest to Strasbourg).

In [63]:
# pip install s3fs
# pip install sparkdl
# pip install keras

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
awscli 1.22.67 requires botocore==1.24.12, but you have botocore 1.23.24 which is incompatible.
Successfully installed aiobotocore-2.1.2 aiohttp-3.8.1 aioitertools-0.10.0 aiosignal-1.2.0 async-timeout-4.0.2 botocore-1.23.24 frozenlist-1.3.0 fsspec-2022.2.0 multidict-6.0.2 s3fs-2022.2.0 typing-extensions-4.1.1 yarl-1.7.2
Note: you may need to restart the kernel to use updated packages.

In [64]:
# import pandas as pd
# import numpy as np
# import pyspark
# from PIL import Image
# import s3fs

# import sys, os #, glob
# from pyspark.sql import SparkSession
# from pyspark.ml.image import ImageSchema
# from pyspark.sql.functions import lit


In [65]:
# fs_test1 = s3fs.S3FileSystem()

# # # it works, but I must comment out this section, because every PUT and LIST request is payed and I'm already above the free limit.

# # To List 5 files in your accessible bucket
# # fs_test1.ls('s3://bucket-name/data/')[:5]
# print('bucket:', fs_test1.ls('s3://p8-s3-bucket')) # ['p8-s3-bucket/Fig']
# # print('some files in bucket:', fs.ls('s3://p8-s3-bucket/Fig')[:5]) # ['p8-s3-bucket/Fig'] # good code just later the folder Fig was removed

In [66]:
# # open it directly

# # it works, but I must comment out this section, because every PUT and LIST request is payed and I'm already above the free limit.

# with fs_test1.open(f'p8-s3-bucket/0_100.jpg') as f:  # good code just explore paths
# # with fs_test1.open(f'p8-s3-bucket/Mango/0_100.jpg') as f: 
#     display(Image.open(f))
#     pic = Image.open(f)
#     pix = np.array(pic.getdata())
#     pixresha = np.array(pic.getdata()).reshape(pic.size[0], pic.size[1], 3)
# print('original', pix.shape, 'reshaped', pixresha.shape)

In [67]:
# sys.path.append('smth')
# sys.path.remove('smth')
# sys.path

## Write the features to csv file

In [68]:
# df.write.csv(os.path.join(tempfile.mkdtemp(), 'data'))
# features_df.write.csv("/home/vb/Desktop/test3", 'overwrite') # AnalysisException: CSV data source does not support array<float> data type

In [69]:
# df_test_structure = spark.read.parquet("test2/fruit_features.parquet")
# df_test_structure.printSchema()
# df_test_structure.show(truncate=True) 

In [70]:
# flattened_df = df_test_structure.withColumn("features", explode("features"))
# flattened_df.printSchema()
# flattened_df.show(truncate=True) 

not the desired result: a new line of PS DataFrame is created for every element in the array.